# Modeling

전처리된 실제 feature table을 직접 읽어서,
split -> candidate 비교 -> 최종 선택 -> test 평가를 notebook 안에서 순서대로 확인합니다.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.svm import NuSVR, SVR


In [2]:
PROJECT_ROOT = Path("/Users/hyun/workspace/data")
RESULTS_DIR = PROJECT_ROOT / "results"

TRAIN_FEATURE_CSV = RESULTS_DIR / "batch1_train_batch2_test_full_q1_q5_elasticnet_train_features.csv"
TEST_FEATURE_CSV = RESULTS_DIR / "batch1_train_batch2_test_full_q1_q5_elasticnet_test_features.csv"

BASE_FEATURES = [
    "Qd_delta_100_10",
    "Qd_retention_100_10",
    "IR_delta_100_10",
    "QC_retention_100_10",
    "chargetime_100_mean",
    "IR_cv_1_100",
    "Qd_QC_ratio_100",
    "Qd_slope_1_100",
    "Qd_drop_per_100",
]

EXTRA_POOL = [
    "policy_soc_pct",
    "policy_charge_c_last",
    "Qd_cv_1_100",
    "Qd_resid_std_1_100",
    "IR_100_mean",
    "Qd_mean_1_100",
    "coulombic_efficiency_100_mean",
    "QC_delta_100_10",
    "Qd_100",
]

MODEL_KEYS = ["svr", "nusvr_a", "nusvr_b", "kr", "ens_nu", "ens_svr_kr"]


In [3]:
batch1 = pd.read_csv(TRAIN_FEATURE_CSV)
batch2 = pd.read_csv(TEST_FEATURE_CSV)

batch1 = batch1[batch1["cycle_life"].notna()].copy().reset_index(drop=True)
batch2 = batch2[batch2["cycle_life"].notna()].copy().reset_index(drop=True)

batch1 = batch1.replace([np.inf, -np.inf], np.nan)
batch2 = batch2.replace([np.inf, -np.inf], np.nan)

pd.Series({
    "train_rows": len(batch1),
    "test_rows": len(batch2),
    "train_cells": batch1["cell_id"].nunique(),
    "test_cells": batch2["cell_id"].nunique(),
})

train_rows     46
test_rows      39
train_cells    46
test_cells     39
dtype: int64

In [4]:
batch1[["cell_id", "cycle_life", "charging_policy", *BASE_FEATURES[:4]]].head()

,cell_id,cycle_life,charging_policy,Qd_delta_100_10,Qd_retention_100_10,IR_delta_100_10,QC_retention_100_10
0,0,1190.0,3.6C(80%)-3.6C,0.001375,1.001280,0.000095,1.001278
1,1,1179.0,3.6C(80%)-3.6C,0.000853,1.000790,0.000129,1.000814
2,2,1177.0,3.6C(80%)-3.6C,0.001159,1.001069,0.000167,1.001024
3,3,1226.0,4C(80%)-4C,0.000442,1.000408,0.000201,1.000442
4,4,1227.0,4C(80%)-4C,0.000692,1.000640,0.000118,1.000655


In [5]:
def mape_pct(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(mean_absolute_percentage_error(y_true, np.clip(y_pred, 1e-6, None)) * 100.0)


def prepare_xy(train_df: pd.DataFrame, test_df: pd.DataFrame, features: list[str], scaler: str) -> tuple[np.ndarray, np.ndarray]:
    imputer = SimpleImputer(strategy="median")
    x_train = imputer.fit_transform(train_df[features])
    x_test = imputer.transform(test_df[features])
    if scaler == "robust":
        scaler_obj = RobustScaler()
        x_train = scaler_obj.fit_transform(x_train)
        x_test = scaler_obj.transform(x_test)
    elif scaler == "standard":
        scaler_obj = StandardScaler()
        x_train = scaler_obj.fit_transform(x_train)
        x_test = scaler_obj.transform(x_test)
    return x_train, x_test


def fit_predict(train_df: pd.DataFrame, test_df: pd.DataFrame, features: list[str], model_key: str) -> np.ndarray:
    scaler = "standard" if model_key == "kr" else "robust"
    x_train, x_test = prepare_xy(train_df, test_df, features, scaler)
    y_train = np.log1p(train_df["cycle_life"].to_numpy(float))
    if model_key == "svr":
        model = SVR(kernel="rbf", C=20.0, epsilon=0.05, gamma="scale")
        model.fit(x_train, y_train)
        return np.expm1(model.predict(x_test))
    if model_key == "nusvr_a":
        model = NuSVR(kernel="rbf", C=30.0, nu=0.25, gamma="scale")
        model.fit(x_train, y_train)
        return np.expm1(model.predict(x_test))
    if model_key == "nusvr_b":
        model = NuSVR(kernel="rbf", C=50.0, nu=0.20, gamma="scale")
        model.fit(x_train, y_train)
        return np.expm1(model.predict(x_test))
    if model_key == "kr":
        model = KernelRidge(alpha=0.1, kernel="rbf", gamma=0.2)
        model.fit(x_train, y_train)
        return np.expm1(model.predict(x_test))
    if model_key == "ens_nu":
        pred_a = fit_predict(train_df, test_df, features, "nusvr_a")
        pred_b = fit_predict(train_df, test_df, features, "nusvr_b")
        return 0.75 * pred_a + 0.25 * pred_b
    if model_key == "ens_svr_kr":
        pred_a = fit_predict(train_df, test_df, features, "svr")
        pred_b = fit_predict(train_df, test_df, features, "kr")
        return 0.5 * pred_a + 0.5 * pred_b
    raise ValueError(model_key)


def cv_best(train_df: pd.DataFrame, features: list[str]) -> dict:
    groups = train_df["charging_policy"].astype(str).to_numpy()
    n_groups = pd.Series(groups).nunique()
    splitter = GroupKFold(n_splits=min(5, max(2, n_groups)))
    best = None
    for model_key in MODEL_KEYS:
        fold_scores = []
        for tr_idx, va_idx in splitter.split(train_df, groups=groups):
            tr = train_df.iloc[tr_idx].copy()
            va = train_df.iloc[va_idx].copy()
            pred = fit_predict(tr, va, features, model_key)
            fold_scores.append(mape_pct(va["cycle_life"].to_numpy(float), pred))
        row = {
            "cv_model_key": model_key,
            "cv_mean_mape_pct": float(np.mean(fold_scores)),
            "cv_std_mape_pct": float(np.std(fold_scores, ddof=0)),
        }
        if best is None or (row["cv_mean_mape_pct"], row["cv_std_mape_pct"]) < (
            best["cv_mean_mape_pct"],
            best["cv_std_mape_pct"],
        ):
            best = row
    return best


def build_candidate_specs() -> list[dict[str, object]]:
    candidates = []
    seen = set()

    def add_candidate(name: str, features: list[str]) -> None:
        key = tuple(sorted(features))
        if key in seen:
            return
        seen.add(key)
        candidates.append({
            "candidate_name": name,
            "features": list(features),
            "feature_count": len(features),
        })

    add_candidate("base", BASE_FEATURES)
    for feature in BASE_FEATURES:
        add_candidate(f"remove:{feature}", [f for f in BASE_FEATURES if f != feature])
    for feature in EXTRA_POOL:
        add_candidate(f"add:{feature}", BASE_FEATURES + [feature])
    for remove_feature in BASE_FEATURES:
        for add_feature in EXTRA_POOL:
            proposal = [f for f in BASE_FEATURES if f != remove_feature] + [add_feature]
            add_candidate(f"swap:{remove_feature}->{add_feature}", proposal)

    return candidates


In [6]:
groups = batch1["charging_policy"].astype(str).to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, valid_idx = next(gss.split(batch1, groups=groups))

train_sub = batch1.iloc[train_idx].copy().reset_index(drop=True)
valid_sub = batch1.iloc[valid_idx].copy().reset_index(drop=True)
candidate_specs = build_candidate_specs()

pd.Series({
    "train_sub_rows": len(train_sub),
    "valid_sub_rows": len(valid_sub),
    "candidate_count": len(candidate_specs),
    "policy_groups_train_sub": train_sub["charging_policy"].nunique(),
    "policy_groups_valid_sub": valid_sub["charging_policy"].nunique(),
})

train_sub_rows              35
valid_sub_rows              11
candidate_count            100
policy_groups_train_sub     18
policy_groups_valid_sub      5
dtype: int64

In [7]:
candidates = []
for spec in candidate_specs:
    features = list(spec["features"])
    cv = cv_best(train_sub, features)
    valid_pred = fit_predict(train_sub, valid_sub, features, cv["cv_model_key"])
    valid = mape_pct(valid_sub["cycle_life"].to_numpy(float), valid_pred)
    candidates.append({
        "candidate_name": spec["candidate_name"],
        "features": features,
        "feature_count": len(features),
        **cv,
        "valid_mape_pct": valid,
    })

cand_df = pd.DataFrame(candidates).sort_values(
    ["valid_mape_pct", "cv_mean_mape_pct", "cv_std_mape_pct", "feature_count"]
).reset_index(drop=True)

cand_df.head(20)

,candidate_name,features,feature_count,cv_model_key,cv_mean_mape_pct,cv_std_mape_pct,valid_mape_pct
0,swap:Qd_drop_per_100->Qd_100,"[Qd_delta_100_10, Qd_retention_100_10, IR_delt...",9,svr,12.553087,3.693482,10.561631
1,swap:Qd_slope_1_100->Qd_100,"[Qd_delta_100_10, Qd_retention_100_10, IR_delt...",9,svr,12.558067,3.689616,10.561755
2,swap:QC_retention_100_10->Qd_100,"[Qd_delta_100_10, Qd_retention_100_10, IR_delt...",9,svr,12.633471,3.856647,10.748810
3,swap:Qd_retention_100_10->Qd_100,"[Qd_delta_100_10, IR_delta_100_10, QC_retentio...",9,svr,12.635068,3.791771,10.787470
4,swap:Qd_delta_100_10->Qd_100,"[Qd_retention_100_10, IR_delta_100_10, QC_rete...",9,svr,12.646646,3.789031,10.793261
5,swap:Qd_drop_per_100->Qd_mean_1_100,"[Qd_delta_100_10, Qd_retention_100_10, IR_delt...",9,svr,12.844527,3.710324,11.042968
6,swap:Qd_slope_1_100->Qd_mean_1_100,"[Qd_delta_100_10, Qd_retention_100_10, IR_delt...",9,svr,12.843931,3.711624,11.043830
7,add:Qd_100,"[Qd_delta_100_10, Qd_retention_100_10, IR_delt...",10,svr,13.250262,4.052864,11.195546
8,swap:Qd_retention_100_10->Qd_mean_1_100,"[Qd_delta_100_10, IR_delta_100_10, QC_retentio...",9,svr,12.934908,3.801596,11.246929
9,swap:Qd_delta_100_10->Qd_mean_1_100,"[Qd_retention_100_10, IR_delta_100_10, QC_rete...",9,svr,12.938507,3.788281,11.250835


In [8]:
best = cand_df.iloc[0].to_dict()
best_features = list(best["features"])
best_model_key = str(best["cv_model_key"])
test_pred = fit_predict(batch1, batch2, best_features, best_model_key)
test_mape = mape_pct(batch2["cycle_life"].to_numpy(float), test_pred)

summary = {
    "selected_candidate": best["candidate_name"],
    "selected_features": best_features,
    "selected_model": best_model_key,
    "train_cv_mape_pct": float(best["cv_mean_mape_pct"]),
    "train_cv_std_mape_pct": float(best["cv_std_mape_pct"]),
    "valid_mape_pct": float(best["valid_mape_pct"]),
    "test_mape_pct": float(test_mape),
    "gap_train_valid_pctp": float(best["valid_mape_pct"] - best["cv_mean_mape_pct"]),
    "gap_valid_test_pctp": float(test_mape - best["valid_mape_pct"]),
    "gap_target_test_pctp": float(test_mape - 9.1),
}

pd.Series(summary)

selected_candidate                            swap:Qd_drop_per_100->Qd_100
selected_features        [Qd_delta_100_10, Qd_retention_100_10, IR_delt...
selected_model                                                         svr
train_cv_mape_pct                                                12.553087
train_cv_std_mape_pct                                             3.693482
valid_mape_pct                                                   10.561631
test_mape_pct                                                     18.72146
gap_train_valid_pctp                                             -1.991456
gap_valid_test_pctp                                               8.159829
gap_target_test_pctp                                               9.62146
dtype: object

In [9]:
pred_df = batch2[["cell_id", "cycle_life", "charging_policy"]].copy()
pred_df["y_pred"] = test_pred
pred_df["ape_pct"] = (pred_df["cycle_life"] - pred_df["y_pred"]).abs() / pred_df["cycle_life"] * 100.0
pred_df.sort_values("ape_pct", ascending=False).head(15)

,cell_id,cycle_life,charging_policy,y_pred,ape_pct
32,34,1186.0,5.6C(26%)-4.5C-newstructure,391.453001,66.993845
35,43,1029.0,4.8C(80%)-4.8C-newstructure,417.257855,59.450160
37,45,997.0,5.6C(26%)-4.5C-newstructure,421.160379,57.757234
8,8,904.0,5.2C(58%)-4C-newstructure,418.778797,53.674912
36,44,841.0,5.2C(58%)-4C-newstructure,417.170925,50.395847
30,32,809.0,4.8C(80%)-4.8C-newstructure,417.175489,48.433190
9,9,791.0,5.6C(26%)-4.5C-newstructure,417.366939,47.235532
7,7,777.0,4.8C(80%)-4.8C-newstructure,417.179008,46.309008
31,33,1140.0,5.2C(58%)-4C-newstructure,769.023260,32.541819
22,24,514.0,4.8C(80%)-4.8C,417.175677,18.837417
